# Introduction to Object-Oriented Programming (OOP) in Python
---
This notebook will guide you step-by-step through the fundamental OOP concepts in Python using a practical example.

We'll cover:
- Classes and objects
- Attributes (class vs instance)
- Encapsulation (private variables)
- Methods (including `@property`, `@setter`)
- Inheritance and `super().__init__`
- Polymorphism and Abstraction
- Class methods and static methods
- Special methods like `__repr__`


## 1. Why OOP?
Before diving into code, let's understand **why we need OOP**.

Imagine building an online store with 100+ products:
- Without OOP: We would create separate variables for each product's name, price, quantity.
- This quickly becomes messy and hard to maintain.

**OOP solves this** by letting us create a blueprint (`class`) and generate multiple products (`objects`) easily.

## Defining a Class
1) A **class** is like a blueprint for creating objects. Think of it as a **template**.
- An **object** is a real instance created from that blueprint.
- It defines what **attributes (data)** and **methods (functions)** the **object** will have.


2) **The `__init__` method** is a **special function** in Python classes known as the **constructor**.

- It is **automatically called** when you create a new object (instance) of the class.
- The primary purpose of `__init__` is to **initialize the object’s attributes** with values provided during the creation of the object.

3) **Defining Attribute Types with `:`**

We can use the colon `:` to define the **type** of an attribute. This is called a **type hint** or **type annotation**.

(self, name`: str`, price`: float`, quantity`=0`)

quantity`=0` has a default value of 0 (implied to be an int).


4) **`Assertions`** are used to **validate inputs** or conditions in the code.

- If an invalid value or condition is encountered (e.g., negative price or quantity), the assertion will **raise an error** and stop the program.
- This helps catch bugs early by ensuring certain conditions are true during execution.


5) **Private Attributes and Name Mangling:**

An **instance attribute** is a variable that belongs to a specific object (instance) of a class.

- Each time you create a new object from a class, it gets **its own copy** of the instance attributes and they are unique to each object.

Attributes with `double underscores` (e.g., ``__name``, ``__price``) are **private (encapsulated)** and hidden from direct access outside the class.

**Private (encapsulated) attributes** protect sensitive data and enforce controlled access using getter/setter methods:
- Use `@property` for controlled **read access**
- Use `@<property>.setter` for controlled **write access**

Although private attributes are hidden, they can still be accessed using **name mangling**:
- Syntax: ``_ClassName__attribute_name``
- Example: ``_Item__price``

This is not recommended for regular use, but it can be useful for debugging or special cases.


6) **Property Decorator for Controlled Access**

- The `@property` decorator allows you to define a **getter method** that lets you access a private attribute, **read-only access**, like a regular attribute (e.g., `item.price` instead of `item.price()`).
  
- Using the **`@property` decorator** makes it act like an attribute (no need to call it like a method).   


- A **setter method** can be created using `@<property_name>.setter` decorator to allow controlled modification of the private attribute.

- This approach enables **controlled access** to private (encapsulated) attributes, allowing validation or additional logic when getting or setting values.

**Analogy:** Bank account balance can only be changed through deposits/withdrawals, not by directly editing the balance.

7) **Class Method and `cls` Parameter:**
- `@classmethod`: Operates on the class itself; often used for **alternative constructors** (e.g., load from CSV).
- A **class method** receives `cls` as its first parameter, which refers to the **class itself**, not an instance.  
- This method can read data from a CSV file and create multiple objects (e.g., multiple `Item` instances).  
- It’s useful for **bulk-creating instances** based on external data sources.




8) **Static Method: Utility Function, No `self` or `cls`**
- `@staticmethod`: Utility function placed inside class for logical grouping; does not access class or object data.

- Does **not** require an object or class reference.  
- This method is utility-like: it's related to the class but doesn't need access to class or instance data.
- Simply checks if a number is an integer or a float that behaves like an integer (e.g., `5.0`).  



9) **Magic Methods**

Magic methods (also called **dunder methods**, short for "double underscore") are special methods that start and end with double underscores, like `__init__`, `__str__`, or `__len__`.

- `print(object)` calls the `__str__()` method if it is defined.
- If `__str__()` is **not** defined, Python will use the `__repr__()` method instead.

Both `__repr__` and `__str__` are **magic methods** used to ***define string representations of an object***.

 - `__repr__` → Developer-Friendly

 - `__str__` → User-Friendly



In [44]:
class Item:
    # Class Attribute (shared among all instances)
    pay_rate = 0.8  # 20% discount
    all = []  # Tracks all created items

    # Constructor method
    def __init__(self, name: str, price: float, quantity=0):
        # Validate input
        assert price >= 0, f"Price {price} must be non-negative!"
        assert quantity >= 0, f"Quantity {quantity} must be non-negative!"

        # Private attributes (Encapsulation)
        self.__name = name
        self.__price = price

        # Public attribute
        self.quantity = quantity

        # Add object to class-level list
        Item.all.append(self)

    # Getter for price
    @property
    def price(self):
        return self.__price

    # Getter and Setter for name with validation
    @property
    def name(self):
      # Getter: returns the private __name attribute
        return self.__name

    @name.setter
    def name(self, value):
      # Setter: allows us to set a new name but with a validation rule
        if len(value) > 10:
            raise Exception("The name is too long!")
        self.__name = value

    # Business logic methods
    def apply_discount(self):
        self.__price = self.__price * self.pay_rate

    def apply_increment(self, increment_value):
        self.__price = self.__price + self.__price * increment_value

    def calculate_total_price(self):
        return self.__price * self.quantity

    # Class Method for alternate object creation
    @classmethod
    def instantiate_from_csv(cls):
        import csv
        with open('items.csv', 'r') as f:
            reader = csv.DictReader(f)
            items = list(reader)

        for item in items:
            cls(
                name=item.get('name'),
                price=float(item.get('price')),
                quantity=int(item.get('quantity')),
            )

    # Static Method (utility)
    # isinstance() is a built-in Python function that checks if a value is of a specific type. It returns True or False.
    @staticmethod
    def is_integer(num):
        if isinstance(num, float):
            return num.is_integer()
        elif isinstance(num, int):
            return True
        return False

    # Special Method for representation
    def __repr__(self):
        return f"{self.__class__.__name__}('{self.name}', {self.__price}, {self.quantity})"


## Inheritance and `super().__init__`

10) **Inheritance** allows a class (**child**) to reuse attributes and methods of another class (**parent**).

Why use `super().__init__`?
- To reuse the parent’s initialization logic.
- Avoids code duplication: if parent `__init__` changes, child automatically benefits.
- Allows child to extend functionality by adding new attributes.

**Step-by-step flow when creating a `Phone` object:**
1. `Phone()` constructor is called.
2. `Phone.__init__` runs and calls `super().__init__`.
3. Parent (`Item.__init__`) initializes name, price, and quantity.
4. Control returns to `Phone.__init__`, which initializes `broken_phones`.
5. Result: Object has **parent attributes + child-specific attributes**.

🚗 Analogy: **Vehicle → Car**  
The parent class sets up the **engine** and **wheels**.  
The child class (Car) uses `super()` to avoid rewriting engine/wheel setup, and adds extra features like **trunk size**.

In [45]:
# Child Class: Phone inherits from Item
class Phone(Item):
    pay_rate = 0.5  # 50% discount for phones

    def __init__(self, name: str, price: float, quantity=0, broken_phones=0):
        # Call parent constructor to initialize name, price, and quantity
        super().__init__(name, price, quantity)

        # Validate and set child-specific attribute
        assert broken_phones >= 0, "Broken Phones must be non-negative!"
        self.broken_phones = broken_phones


In [46]:

# Another Child Class: Keyboard inherits from Item
class Keyboard(Item):
    pay_rate = 0.7  # 30% discount

    def __init__(self, name: str, price: float, quantity=0):
        # Reuse parent initialization
        super().__init__(name, price, quantity)


### Example Usage


In [48]:
item1 = Item("Book", 15.99, 3)
phone1 = Phone("iPhone", 999.99, 2, broken_phones=1)
keyboard1 = Keyboard("Logitech", 49.99, 5)

print(item1.calculate_total_price())
print(phone1.calculate_total_price())
print(f"{keyboard1.calculate_total_price():.2f}")


47.97
1999.98
249.95


## 4. Polymorphism and Abstraction
- **Polymorphism:** Same method (`apply_discount`) behaves differently for different classes due to different `pay_rate` values.
- **Abstraction:** User just calls `apply_discount()`; they don’t need to know the internal calculation.

In [49]:
item1 = Item("Book", 15.99, 3)
phone1 = Phone("iPhone", 999.99, 2, broken_phones=1)
keyboard1 = Keyboard("Logitech", 49.99, 5)

item1.apply_discount()
print(round(item1.calculate_total_price(), 2))

phone1.apply_increment(0.2)
print(phone1.calculate_total_price())

keyboard1.apply_discount()
print(keyboard1.calculate_total_price())


38.38
2399.976
174.965


In [50]:

# Create Keyboard object/ apply discout / compare the price befor and after.
keyboard = Keyboard("GamingKB", 1000, 3)
print(keyboard)
print(f"Original Price: {keyboard.price}")

keyboard.apply_discount()   # It does not return any value, just update the price.
print(f"Price after discount: {keyboard.price}")
print(keyboard)  # it runs __repr__()

# Create Phone object / apply increment / compare the price befor and after.
phone = Phone("iPhone", 1200, 2, 1)
print(phone)
print(f"Original Price: {phone.price}")
phone.apply_increment(0.15)
print(f"Price after Incrementing: {phone.price}")
print(phone)


Keyboard('GamingKB', 1000, 3)
Original Price: 1000
Price after discount: 700.0
Keyboard('GamingKB', 700.0, 3)
Phone('iPhone', 1200, 2)
Original Price: 1200
Price after Incrementing: 1380.0
Phone('iPhone', 1380.0, 2)


In [51]:
print(f"object's original name: {item1.name}")
item1.name = "Notebook"
print(f"object's new name: {item1.name}")

#price is a private attribute and through "Setter" we can set a new name but with a validation rule, it can not be longer than 10 chr.
try:
    item1.name = "VeryLongItemName"
except Exception as e:
    print(e)

object's original name: Book
object's new name: Notebook
The name is too long!


In [53]:

# Example Static Method usage
print(Item.is_integer(10))
print(Item.is_integer(10.0))      # True (because it's a float with no decimal part)
print(Item.is_integer(10.5))
print(Item.is_integer("10"))


True
True
False
False
